In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 08_final_model_lock_before_test.ipynb
# COMPLETE SINGLE-CELL CODE
# ============================================================
#
# PURPOSE
# -------
# Create an immutable PRE-TEST lock for Route A.
#
# THIS NOTEBOOK:
#   - performs NO training
#   - performs NO inference
#   - defines NO official test path
#   - loads NO official test data
#   - evaluates NO official test data
#
# It freezes:
#   1. primary model = Swin
#   2. comparator = ViT
#   3. exact 3 seeds x 5 folds = 15 checkpoints/backbone
#   4. exact checkpoint file hashes
#   5. equal-weight checkpoint probability ensemble
#   6. horizontal-flip TTA
#   7. 384 x 384 deterministic evaluation transform
#   8. modal_low derived patient reference
#   9. mean_probability_argmax patient aggregation
#  10. primary and secondary metrics
#  11. paired patient bootstrap analysis
#  12. per-class uncertainty analysis
#  13. predefined sensitivity analyses
#  14. Route A scope: Swin vs ViT only
#
# After this notebook completes:
#   DO NOT modify any locked inference or analysis decision
#   after official test access.
#
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

import os
import json
import hashlib
import platform
import shutil

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. PRE-TEST SAFETY CHECK
# ============================================================

FORBIDDEN_TEST_OBJECTS = [
    "TEST_PATH",
    "TEST_ROOT",
    "TEST_DIR",
    "OFFICIAL_TEST_PATH",
    "OFFICIAL_TEST_ROOT",
    "official_test_df",
    "test_df",
    "test_dataset",
    "test_loader",
    "test_predictions",
    "test_patient_predictions",
    "test_image_predictions",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "ensemble_test_metrics",
    "ensemble_test_patient_metrics",
]

present_test_objects = [
    object_name
    for object_name in FORBIDDEN_TEST_OBJECTS
    if object_name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects already exist in this runtime:\n"
        f"{present_test_objects}\n\n"
        "Restart the runtime and run ONLY Notebook 08 before "
        "opening or defining the official test partition."
    )

print("=" * 120)
print("PRE-TEST SAFETY CHECK PASSED")
print("=" * 120)

print(
    "Official LIMUC test path defined:",
    False,
)

print(
    "Official LIMUC test data loaded:",
    False,
)

print(
    "Official LIMUC test data evaluated:",
    False,
)

print(
    "\nNo training or inference will be performed in this notebook."
)


# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 3. Source paths
# ============================================================

REVISION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
)

NOTEBOOK01_ROOT = (
    REVISION_ROOT
    / "01_vit_loss_screening_development_only"
)

NOTEBOOK02_ROOT = (
    REVISION_ROOT
    / "02_vit_loss_multiseed_confirmation_epoch_resume_v2"
)

NOTEBOOK03_ROOT = (
    REVISION_ROOT
    / "03_patient_aggregation_and_reference_sensitivity"
)

NOTEBOOK04_ROOT = (
    REVISION_ROOT
    / "04_swin_loss_screening_development_only"
)

NOTEBOOK05_ROOT = (
    REVISION_ROOT
    / "05_swin_loss_multiseed_confirmation_uniform_runtime_v1"
)

NOTEBOOK06_ROOT = (
    REVISION_ROOT
    / "06_backbone_benchmark_development_only_v1"
)

NOTEBOOK07_ROOT = (
    REVISION_ROOT
    / "07_locked_swin_ablation_development_only_v1"
)

OUTPUT_ROOT = (
    REVISION_ROOT
    / "08_final_model_lock_before_test"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 4. Important source artifacts
# ============================================================

MANIFEST_PATH = (
    NOTEBOOK01_ROOT
    / "development_image_manifest_with_folds.csv"
)

FOLD_ASSIGNMENT_PATH = (
    NOTEBOOK01_ROOT
    / "patient_fold_assignment.csv"
)

REFERENCE_AUDIT_PATH = (
    NOTEBOOK01_ROOT
    / "patient_reference_rule_sensitivity.csv"
)

NOTEBOOK03_PLAN_PATH = (
    NOTEBOOK03_ROOT
    / "analysis_plan_frozen_before_aggregation.json"
)

LOCKED_AGGREGATION_PATH = (
    NOTEBOOK03_ROOT
    / "locked_patient_aggregation_rule.json"
)

NOTEBOOK05_PLAN_PATH = (
    NOTEBOOK05_ROOT
    / "analysis_plan_frozen_before_confirmation.json"
)

LOCKED_SWIN_CONFIGURATION_PATH = (
    NOTEBOOK05_ROOT
    / "locked_representative_swin_configuration.json"
)

NOTEBOOK06_PLAN_PATH = (
    NOTEBOOK06_ROOT
    / "analysis_plan_frozen_before_backbone_benchmark.json"
)

NOTEBOOK06_RESULTS_PATH = (
    NOTEBOOK06_ROOT
    / "backbone_results_by_seed.csv"
)

NOTEBOOK06_SUMMARY_PATH = (
    NOTEBOOK06_ROOT
    / "backbone_multiseed_summary.csv"
)

NOTEBOOK06_RANKING_PATH = (
    NOTEBOOK06_ROOT
    / "backbone_multiseed_ranking.csv"
)

NOTEBOOK06_BOOTSTRAP_PATH = (
    NOTEBOOK06_ROOT
    / "backbone_paired_bootstrap_summary.csv"
)

LOCKED_BACKBONE_PATH = (
    NOTEBOOK06_ROOT
    / "locked_representative_backbone.json"
)

NOTEBOOK06_DECISION_PATH = (
    NOTEBOOK06_ROOT
    / "backbone_benchmark_decision_summary.json"
)

NOTEBOOK07_PLAN_PATH = (
    NOTEBOOK07_ROOT
    / "analysis_plan_frozen_before_ablation_training.json"
)

NOTEBOOK07_RESULTS_PATH = (
    NOTEBOOK07_ROOT
    / "ablation_multiseed_summary.csv"
)

NOTEBOOK07_CONTRAST_PATH = (
    NOTEBOOK07_ROOT
    / "ablation_predeclared_contrasts_paired_bootstrap.csv"
)

NOTEBOOK07_PRIMARY_LOCK_PATH = (
    NOTEBOOK07_ROOT
    / "primary_model_lock_confirmed_after_ablation.json"
)

NOTEBOOK07_DECISION_PATH = (
    NOTEBOOK07_ROOT
    / "ablation_decision_summary.json"
)


REQUIRED_SOURCE_PATHS = [
    MANIFEST_PATH,
    FOLD_ASSIGNMENT_PATH,
    REFERENCE_AUDIT_PATH,

    NOTEBOOK03_PLAN_PATH,
    LOCKED_AGGREGATION_PATH,

    NOTEBOOK05_PLAN_PATH,
    LOCKED_SWIN_CONFIGURATION_PATH,

    NOTEBOOK06_PLAN_PATH,
    NOTEBOOK06_RESULTS_PATH,
    NOTEBOOK06_SUMMARY_PATH,
    NOTEBOOK06_RANKING_PATH,
    NOTEBOOK06_BOOTSTRAP_PATH,
    LOCKED_BACKBONE_PATH,
    NOTEBOOK06_DECISION_PATH,

    NOTEBOOK07_PLAN_PATH,
    NOTEBOOK07_RESULTS_PATH,
    NOTEBOOK07_CONTRAST_PATH,
    NOTEBOOK07_PRIMARY_LOCK_PATH,
    NOTEBOOK07_DECISION_PATH,
]

for required_path in REQUIRED_SOURCE_PATHS:
    if not required_path.exists():
        raise FileNotFoundError(
            "Required pre-test source artifact is missing:\n"
            f"{required_path}"
        )

print("\nAll required development-only source artifacts exist.")


# ============================================================
# 5. Generic helpers
# ============================================================

def temporary_path(
    path: Path,
) -> Path:
    return path.with_name(
        path.name + ".temporary"
    )


def atomic_json_save(
    payload,
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            payload,
            file,
            indent=2,
            sort_keys=False,
            default=str,
        )

        file.flush()

        os.fsync(
            file.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def atomic_csv_save(
    dataframe,
    path: Path,
    index=False,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    dataframe.to_csv(
        temp_path,
        index=index,
    )

    os.replace(
        temp_path,
        path,
    )


def atomic_text_save(
    text,
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = temporary_path(
        path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            text
        )

        file.flush()

        os.fsync(
            file.fileno()
        )

    os.replace(
        temp_path,
        path,
    )


def file_sha256(
    path: Path,
    block_size=8 * 1024 * 1024,
):
    hasher = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            block = file.read(
                block_size
            )

            if not block:
                break

            hasher.update(
                block
            )

    return hasher.hexdigest()


def dictionary_sha256(
    dictionary,
):
    encoded = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        encoded
    ).hexdigest()


def safe_package_version(
    package_name,
):
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


# Remove abandoned temporary lock files.
for temporary_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        temporary_file.unlink()

    except Exception:
        pass


# ============================================================
# 6. Load frozen source decisions
# ============================================================

with open(
    LOCKED_AGGREGATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_aggregation = json.load(
        file
    )

with open(
    LOCKED_SWIN_CONFIGURATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_swin_confirmation = json.load(
        file
    )

with open(
    NOTEBOOK06_PLAN_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook06_plan = json.load(
        file
    )

with open(
    LOCKED_BACKBONE_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_backbone = json.load(
        file
    )

with open(
    NOTEBOOK06_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook06_decision = json.load(
        file
    )

with open(
    NOTEBOOK07_PRIMARY_LOCK_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook07_primary_lock = json.load(
        file
    )

with open(
    NOTEBOOK07_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    notebook07_decision = json.load(
        file
    )


# ============================================================
# 7. Validate frozen primary decisions
# ============================================================

PRIMARY_REFERENCE_RULE = (
    "modal_low"
)

LOCKED_PATIENT_AGGREGATION = (
    "mean_probability_argmax"
)

PRIMARY_BACKBONE = (
    "Swin"
)

COMPARATOR_BACKBONE = (
    "ViT"
)

EXPECTED_PRIMARY_CONFIGURATION = (
    "swin_L3_expected_ordinal_distance_a1_w010"
)

EXPECTED_COMPARATOR_CONFIGURATION = (
    "vit_L5_expected_ordinal_distance_a1_w020"
)

SEEDS = [
    42,
    2025,
    2026,
]

FOLDS = [
    0,
    1,
    2,
    3,
    4,
]


if (
    locked_aggregation.get(
        "locked_aggregation_rule"
    )
    !=
    LOCKED_PATIENT_AGGREGATION
):
    raise RuntimeError(
        "Patient aggregation differs from the "
        "Notebook 03 lock."
    )


if (
    locked_aggregation.get(
        "primary_reference_rule"
    )
    !=
    PRIMARY_REFERENCE_RULE
):
    raise RuntimeError(
        "Primary patient reference differs from modal_low."
    )


if (
    locked_backbone.get(
        "locked_representative_backbone"
    )
    !=
    PRIMARY_BACKBONE
):
    raise RuntimeError(
        "Notebook 06 did not lock Swin as the "
        "representative Route A backbone."
    )


primary_configuration = (
    locked_backbone[
        "locked_configuration"
    ]
)


if (
    primary_configuration.get(
        "representative_configuration"
    )
    !=
    EXPECTED_PRIMARY_CONFIGURATION
):
    raise RuntimeError(
        "Unexpected primary Swin configuration."
    )


if (
    notebook07_primary_lock.get(
        "primary_model_changed_by_ablation"
    )
    is not False
):
    raise RuntimeError(
        "Notebook 07 indicates that the primary model "
        "was changed by the ablation."
    )


if (
    notebook07_primary_lock.get(
        "full_model_still_locked"
    )
    is not True
):
    raise RuntimeError(
        "Notebook 07 does not confirm that the full "
        "primary model remained locked."
    )


if (
    notebook07_decision.get(
        "primary_model_selection_performed"
    )
    is not False
):
    raise RuntimeError(
        "Notebook 07 unexpectedly performed model selection."
    )


print("\nFrozen development decisions validated.")

print(
    "Primary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Patient aggregation:",
    LOCKED_PATIENT_AGGREGATION,
)

print(
    "Primary backbone:",
    PRIMARY_BACKBONE,
)

print(
    "Primary configuration:",
    EXPECTED_PRIMARY_CONFIGURATION,
)

print(
    "Comparator backbone:",
    COMPARATOR_BACKBONE,
)

print(
    "Comparator configuration:",
    EXPECTED_COMPARATOR_CONFIGURATION,
)


# ============================================================
# 8. Extract exact Route A backbone configurations
# ============================================================

backbone_configurations = (
    notebook06_plan[
        "backbone_configurations"
    ]
)

if (
    PRIMARY_BACKBONE
    not in backbone_configurations
):
    raise RuntimeError(
        "Swin configuration missing from Notebook 06 plan."
    )

if (
    COMPARATOR_BACKBONE
    not in backbone_configurations
):
    raise RuntimeError(
        "ViT configuration missing from Notebook 06 plan."
    )


SWIN_CONFIG = copy_swin_config = dict(
    backbone_configurations[
        PRIMARY_BACKBONE
    ]
)

VIT_CONFIG = copy_vit_config = dict(
    backbone_configurations[
        COMPARATOR_BACKBONE
    ]
)


if (
    SWIN_CONFIG[
        "representative_configuration"
    ]
    !=
    EXPECTED_PRIMARY_CONFIGURATION
):
    raise RuntimeError(
        "Swin configuration mismatch."
    )


if (
    VIT_CONFIG[
        "representative_configuration"
    ]
    !=
    EXPECTED_COMPARATOR_CONFIGURATION
):
    raise RuntimeError(
        "ViT configuration mismatch."
    )


SWIN_MODEL_NAME = str(
    SWIN_CONFIG[
        "model_name"
    ]
)

VIT_MODEL_NAME = str(
    VIT_CONFIG[
        "model_name"
    ]
)


print(
    "\nPrimary model identifier:",
    SWIN_MODEL_NAME,
)

print(
    "Comparator model identifier:",
    VIT_MODEL_NAME,
)


# ============================================================
# 9. Validate development dataset and frozen folds
# ============================================================

development_manifest_df = pd.read_csv(
    MANIFEST_PATH
)

required_manifest_columns = {
    "relative_path",
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
}

missing_manifest_columns = (
    required_manifest_columns
    -
    set(
        development_manifest_df.columns
    )
)

if missing_manifest_columns:
    raise RuntimeError(
        "Development manifest missing columns:\n"
        f"{sorted(missing_manifest_columns)}"
    )


for column in [
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
]:
    development_manifest_df[
        column
    ] = (
        development_manifest_df[
            column
        ]
        .astype(
            int
        )
    )


if len(
    development_manifest_df
) != 9590:
    raise RuntimeError(
        "Expected 9,590 development images."
    )


if (
    development_manifest_df[
        "patient_id"
    ].nunique()
    !=
    478
):
    raise RuntimeError(
        "Expected 478 development patients."
    )


if (
    development_manifest_df
    .groupby(
        "patient_id"
    )[
        "cv_fold"
    ]
    .nunique()
    .max()
    !=
    1
):
    raise RuntimeError(
        "A development patient appears in multiple folds."
    )


development_patient_df = (
    development_manifest_df[
        [
            "patient_id",
            "patient_y_true",
            "cv_fold",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)


development_class_counts = (
    development_patient_df[
        "patient_y_true"
    ]
    .value_counts()
    .reindex(
        range(
            4
        ),
        fill_value=0,
    )
    .sort_index()
    .to_dict()
)


development_fold_counts = (
    development_patient_df[
        "cv_fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)


print(
    "\nDevelopment patient class counts:",
    development_class_counts,
)

print(
    "Development fold counts:",
    development_fold_counts,
)


# ============================================================
# 10. Validate Notebook 06 benchmark result
# ============================================================

backbone_ranking_df = pd.read_csv(
    NOTEBOOK06_RANKING_PATH
)

if (
    backbone_ranking_df.iloc[
        0
    ][
        "backbone_name"
    ]
    !=
    PRIMARY_BACKBONE
):
    raise RuntimeError(
        "Notebook 06 ranking does not place Swin first."
    )


notebook06_summary_df = pd.read_csv(
    NOTEBOOK06_SUMMARY_PATH
)

notebook06_bootstrap_df = pd.read_csv(
    NOTEBOOK06_BOOTSTRAP_PATH
)


swin_summary_row = (
    notebook06_summary_df[
        notebook06_summary_df[
            "backbone_name"
        ]
        ==
        "Swin"
    ]
    .iloc[
        0
    ]
)


vit_summary_row = (
    notebook06_summary_df[
        notebook06_summary_df[
            "backbone_name"
        ]
        ==
        "ViT"
    ]
    .iloc[
        0
    ]
)


print("\nNotebook 06 benchmark lock validated.")

print(
    "Swin mean patient QWK:",
    float(
        swin_summary_row[
            "patient_qwk_mean"
        ]
    ),
)

print(
    "ViT mean patient QWK:",
    float(
        vit_summary_row[
            "patient_qwk_mean"
        ]
    ),
)


# ============================================================
# 11. Validate Notebook 07 did not alter the primary model
# ============================================================

ablation_summary_df = pd.read_csv(
    NOTEBOOK07_RESULTS_PATH
)

a0_row = (
    ablation_summary_df[
        ablation_summary_df[
            "condition_name"
        ]
        ==
        "A0_full_locked"
    ]
    .iloc[
        0
    ]
)


if not np.isclose(
    float(
        a0_row[
            "patient_qwk_mean"
        ]
    ),
    float(
        swin_summary_row[
            "patient_qwk_mean"
        ]
    ),
    atol=1e-10,
):
    raise RuntimeError(
        "Notebook 07 A0 QWK does not match "
        "Notebook 06 locked Swin QWK."
    )


if not np.isclose(
    float(
        a0_row[
            "patient_mae_mean"
        ]
    ),
    float(
        swin_summary_row[
            "patient_mae_mean"
        ]
    ),
    atol=1e-10,
):
    raise RuntimeError(
        "Notebook 07 A0 MAE does not match "
        "Notebook 06 locked Swin MAE."
    )


print(
    "\nNotebook 07 full-model results match "
    "Notebook 06 locked Swin results."
)


# ============================================================
# 12. Build exact 30-checkpoint manifest
# ============================================================

CHECKPOINT_MANIFEST_ROWS = []


BACKBONE_EXPECTATIONS = {
    "Swin": {
        "model_name": (
            SWIN_MODEL_NAME
        ),

        "representative_configuration": (
            EXPECTED_PRIMARY_CONFIGURATION
        ),

        "role": (
            "primary"
        ),
    },

    "ViT": {
        "model_name": (
            VIT_MODEL_NAME
        ),

        "representative_configuration": (
            EXPECTED_COMPARATOR_CONFIGURATION
        ),

        "role": (
            "prespecified_comparator"
        ),
    },
}


print(
    "\nHashing exact fold checkpoints."
)

print(
    "This can take several minutes because "
    "all 30 checkpoint files are hashed."
)


for backbone_name in [
    "Swin",
    "ViT",
]:
    expectation = (
        BACKBONE_EXPECTATIONS[
            backbone_name
        ]
    )

    for training_seed in SEEDS:
        for fold in FOLDS:
            fold_directory = (
                NOTEBOOK06_ROOT
                /
                backbone_name
                /
                f"seed_{training_seed}"
                /
                f"fold_{fold}"
            )

            checkpoint_path = (
                fold_directory
                /
                "best_checkpoint.pt"
            )

            fold_result_path = (
                fold_directory
                /
                "fold_result.json"
            )

            validation_image_prediction_path = (
                fold_directory
                /
                "validation_image_predictions.csv"
            )

            for required_path in [
                checkpoint_path,
                fold_result_path,
                validation_image_prediction_path,
            ]:
                if not required_path.exists():
                    raise FileNotFoundError(
                        "Required Notebook 06 fold artifact "
                        "is missing:\n"
                        f"{required_path}"
                    )

            with open(
                fold_result_path,
                "r",
                encoding="utf-8",
            ) as file:
                fold_result = json.load(
                    file
                )

            if (
                fold_result[
                    "backbone_name"
                ]
                !=
                backbone_name
            ):
                raise RuntimeError(
                    "Backbone metadata mismatch:\n"
                    f"{fold_result_path}"
                )

            if (
                int(
                    fold_result[
                        "training_seed"
                    ]
                )
                !=
                int(
                    training_seed
                )
            ):
                raise RuntimeError(
                    "Training-seed metadata mismatch:\n"
                    f"{fold_result_path}"
                )

            if (
                int(
                    fold_result[
                        "fold"
                    ]
                )
                !=
                int(
                    fold
                )
            ):
                raise RuntimeError(
                    "Fold metadata mismatch:\n"
                    f"{fold_result_path}"
                )

            if (
                fold_result[
                    "model_name"
                ]
                !=
                expectation[
                    "model_name"
                ]
            ):
                raise RuntimeError(
                    "Model identifier mismatch:\n"
                    f"{fold_result_path}"
                )

            if (
                fold_result[
                    "representative_configuration"
                ]
                !=
                expectation[
                    "representative_configuration"
                ]
            ):
                raise RuntimeError(
                    "Representative configuration mismatch:\n"
                    f"{fold_result_path}"
                )

            checkpoint_hash = (
                file_sha256(
                    checkpoint_path
                )
            )

            checkpoint_size = int(
                checkpoint_path.stat().st_size
            )

            CHECKPOINT_MANIFEST_ROWS.append({
                "role": (
                    expectation[
                        "role"
                    ]
                ),

                "backbone_name": (
                    backbone_name
                ),

                "model_name": (
                    expectation[
                        "model_name"
                    ]
                ),

                "representative_configuration": (
                    expectation[
                        "representative_configuration"
                    ]
                ),

                "training_seed": (
                    int(
                        training_seed
                    )
                ),

                "fold": (
                    int(
                        fold
                    )
                ),

                "best_epoch": (
                    int(
                        fold_result[
                            "best_epoch"
                        ]
                    )
                ),

                "parameter_count": (
                    int(
                        fold_result[
                            "parameter_count"
                        ]
                    )
                ),

                "fold_fingerprint": (
                    fold_result[
                        "fold_fingerprint"
                    ]
                ),

                "checkpoint_path": str(
                    checkpoint_path
                ),

                "checkpoint_sha256": (
                    checkpoint_hash
                ),

                "checkpoint_size_bytes": (
                    checkpoint_size
                ),

                "fold_result_path": str(
                    fold_result_path
                ),

                "fold_result_sha256": (
                    file_sha256(
                        fold_result_path
                    )
                ),

                "validation_prediction_path": str(
                    validation_image_prediction_path
                ),

                "validation_prediction_sha256": (
                    file_sha256(
                        validation_image_prediction_path
                    )
                ),
            })

            print(
                f"  hashed {backbone_name} | "
                f"seed {training_seed} | fold {fold}"
            )


checkpoint_manifest_df = pd.DataFrame(
    CHECKPOINT_MANIFEST_ROWS
)


if len(
    checkpoint_manifest_df
) != 30:
    raise RuntimeError(
        "Exactly 30 locked checkpoints were expected."
    )


checkpoint_count_check = (
    checkpoint_manifest_df
    .groupby(
        "backbone_name"
    )
    .size()
    .to_dict()
)


if (
    checkpoint_count_check.get(
        "Swin"
    )
    !=
    15
):
    raise RuntimeError(
        "Expected exactly 15 Swin checkpoints."
    )


if (
    checkpoint_count_check.get(
        "ViT"
    )
    !=
    15
):
    raise RuntimeError(
        "Expected exactly 15 ViT checkpoints."
    )


if (
    checkpoint_manifest_df[
        "checkpoint_sha256"
    ]
    .isna()
    .any()
):
    raise RuntimeError(
        "A checkpoint hash is missing."
    )


checkpoint_manifest_path = (
    OUTPUT_ROOT
    /
    "locked_checkpoint_manifest.csv"
)

atomic_csv_save(
    checkpoint_manifest_df,

    checkpoint_manifest_path,

    index=False,
)


print(
    "\nCheckpoint manifest complete:"
)

print(
    checkpoint_count_check
)


# ============================================================
# 13. Build source-artifact provenance manifest
# ============================================================

SOURCE_ARTIFACTS = {
    "development_manifest": (
        MANIFEST_PATH
    ),

    "patient_fold_assignment": (
        FOLD_ASSIGNMENT_PATH
    ),

    "patient_reference_audit": (
        REFERENCE_AUDIT_PATH
    ),

    "aggregation_analysis_plan": (
        NOTEBOOK03_PLAN_PATH
    ),

    "locked_patient_aggregation": (
        LOCKED_AGGREGATION_PATH
    ),

    "swin_confirmation_plan": (
        NOTEBOOK05_PLAN_PATH
    ),

    "locked_swin_confirmation": (
        LOCKED_SWIN_CONFIGURATION_PATH
    ),

    "backbone_benchmark_plan": (
        NOTEBOOK06_PLAN_PATH
    ),

    "backbone_results_by_seed": (
        NOTEBOOK06_RESULTS_PATH
    ),

    "backbone_multiseed_summary": (
        NOTEBOOK06_SUMMARY_PATH
    ),

    "backbone_ranking": (
        NOTEBOOK06_RANKING_PATH
    ),

    "backbone_paired_bootstrap": (
        NOTEBOOK06_BOOTSTRAP_PATH
    ),

    "locked_backbone": (
        LOCKED_BACKBONE_PATH
    ),

    "backbone_decision_summary": (
        NOTEBOOK06_DECISION_PATH
    ),

    "ablation_analysis_plan": (
        NOTEBOOK07_PLAN_PATH
    ),

    "ablation_multiseed_summary": (
        NOTEBOOK07_RESULTS_PATH
    ),

    "ablation_paired_contrasts": (
        NOTEBOOK07_CONTRAST_PATH
    ),

    "ablation_primary_lock": (
        NOTEBOOK07_PRIMARY_LOCK_PATH
    ),

    "ablation_decision_summary": (
        NOTEBOOK07_DECISION_PATH
    ),
}


source_artifact_rows = []

for artifact_name, artifact_path in (
    SOURCE_ARTIFACTS.items()
):
    source_artifact_rows.append({
        "artifact_name": (
            artifact_name
        ),

        "path": str(
            artifact_path
        ),

        "sha256": (
            file_sha256(
                artifact_path
            )
        ),

        "size_bytes": (
            int(
                artifact_path.stat().st_size
            )
        ),
    })


source_artifact_manifest_df = (
    pd.DataFrame(
        source_artifact_rows
    )
)


source_artifact_manifest_path = (
    OUTPUT_ROOT
    /
    "source_artifact_provenance_manifest.csv"
)

atomic_csv_save(
    source_artifact_manifest_df,

    source_artifact_manifest_path,

    index=False,
)


# ============================================================
# 14. Freeze exact test-time preprocessing
# ============================================================

EVALUATION_PREPROCESSING = {
    "color_mode": (
        "RGB"
    ),

    "image_size": [
        384,
        384,
    ],

    "resize_method": (
        "torchvision.transforms.Resize((384, 384))"
    ),

    "center_crop": False,

    "random_crop": False,

    "random_rotation": False,

    "color_jitter": False,

    "gaussian_blur": False,

    "tensor_conversion": (
        "torchvision.transforms.ToTensor()"
    ),

    "normalization_mean": [
        0.485,
        0.456,
        0.406,
    ],

    "normalization_std": [
        0.229,
        0.224,
        0.225,
    ],

    "test_time_augmentation": {
        "enabled": True,

        "views": [
            "original",
            "horizontal_flip",
        ],

        "horizontal_flip_dimension": (
            "image width"
        ),

        "per_checkpoint_combination": (
            "arithmetic mean of the two softmax "
            "probability vectors"
        ),
    },
}


# ============================================================
# 15. Freeze exact model-ensemble inference
# ============================================================

PRIMARY_INFERENCE_SYSTEM = {
    "system_name": (
        "RouteA_Primary_Swin_15Checkpoint_Probability_Ensemble"
    ),

    "role": (
        "primary"
    ),

    "backbone": (
        "Swin"
    ),

    "model_name": (
        SWIN_MODEL_NAME
    ),

    "representative_configuration": (
        EXPECTED_PRIMARY_CONFIGURATION
    ),

    "training_seeds": (
        SEEDS
    ),

    "folds": (
        FOLDS
    ),

    "number_of_checkpoints": (
        15
    ),

    "checkpoint_source": (
        "Notebook 06 uniform development-only "
        "backbone benchmark"
    ),

    "checkpoint_weighting": (
        "equal"
    ),

    "seed_weighting": (
        "equal"
    ),

    "fold_weighting": (
        "equal"
    ),

    "checkpoint_reselection_on_test": (
        False
    ),

    "best_seed_selection_on_test": (
        False
    ),

    "calibration_on_test": (
        False
    ),

    "threshold_tuning_on_test": (
        False
    ),

    "logit_adjustment_on_test": (
        False
    ),

    "probability_ensemble_formula": (
        "For each image x, each checkpoint m produces "
        "TTA-averaged softmax probabilities p_m(x). "
        "The final image probability vector is "
        "p_bar(x) = (1/15) * sum_{m=1}^{15} p_m(x)."
    ),

    "image_prediction_rule": (
        "argmax of the 15-checkpoint mean "
        "probability vector"
    ),
}


COMPARATOR_INFERENCE_SYSTEM = {
    "system_name": (
        "RouteA_Comparator_ViT_15Checkpoint_Probability_Ensemble"
    ),

    "role": (
        "prespecified_comparator"
    ),

    "backbone": (
        "ViT"
    ),

    "model_name": (
        VIT_MODEL_NAME
    ),

    "representative_configuration": (
        EXPECTED_COMPARATOR_CONFIGURATION
    ),

    "training_seeds": (
        SEEDS
    ),

    "folds": (
        FOLDS
    ),

    "number_of_checkpoints": (
        15
    ),

    "checkpoint_source": (
        "Notebook 06 uniform development-only "
        "backbone benchmark"
    ),

    "checkpoint_weighting": (
        "equal"
    ),

    "seed_weighting": (
        "equal"
    ),

    "fold_weighting": (
        "equal"
    ),

    "checkpoint_reselection_on_test": (
        False
    ),

    "best_seed_selection_on_test": (
        False
    ),

    "calibration_on_test": (
        False
    ),

    "threshold_tuning_on_test": (
        False
    ),

    "logit_adjustment_on_test": (
        False
    ),

    "probability_ensemble_formula": (
        "For each image x, each checkpoint m produces "
        "TTA-averaged softmax probabilities p_m(x). "
        "The final image probability vector is "
        "p_bar(x) = (1/15) * sum_{m=1}^{15} p_m(x)."
    ),

    "image_prediction_rule": (
        "argmax of the 15-checkpoint mean "
        "probability vector"
    ),
}


# ============================================================
# 16. Freeze patient-level reference construction
# ============================================================

PATIENT_REFERENCE_PLAN = {
    "primary_reference_name": (
        "modal_low"
    ),

    "source": (
        "official test image-level MES labels"
    ),

    "construction": (
        "For each patient, select the modal image-level "
        "MES label. If multiple MES classes are tied for "
        "the mode, select the lower class."
    ),

    "independently_adjudicated_exam_ground_truth": (
        False
    ),

    "terminology": (
        "derived patient-level reference"
    ),

    "forbidden_primary_terminology": [
        "independent patient-level ground truth",
        "independently adjudicated examination-level ground truth",
    ],

    "alternative_reference_sensitivity": {
        "used_for_primary_model_selection": (
            False
        ),

        "rules": [
            "modal_high",
            "median_low",
            "median_high",
            "maximum_label",
        ],

        "role": (
            "descriptive sensitivity analysis only"
        ),
    },
}


# ============================================================
# 17. Freeze patient-level prediction aggregation
# ============================================================

PATIENT_AGGREGATION_PLAN = {
    "primary_rule": (
        "mean_probability_argmax"
    ),

    "formula": (
        "For patient p with n_p images, first calculate "
        "the final 15-checkpoint image probability vector "
        "for every image. Then compute the arithmetic mean "
        "probability for each class across the patient's "
        "images. The patient prediction is the argmax of "
        "the patient mean probability vector."
    ),

    "equation": (
        "p_bar_patient(c) = (1/n_p) * "
        "sum_i p_bar_image_i(c); "
        "y_hat_patient = argmax_c p_bar_patient(c)"
    ),

    "patient_image_weighting": (
        "equal weight per image"
    ),

    "aggregation_reselection_on_test": (
        False
    ),

    "primary_rule_can_change_after_test_access": (
        False
    ),

    "alternative_aggregation_sensitivity": {
        "used_for_primary_model_selection": (
            False
        ),

        "role": (
            "secondary sensitivity analysis only"
        ),

        "rules": [
            "median_probability_argmax",
            "majority_vote_low_tie",
            "mean_expected_mes_round",
            "median_expected_mes_round",
            "maximum_predicted_class",
        ],
    },
}


# ============================================================
# 18. Freeze primary and secondary evaluation outcomes
# ============================================================

METRIC_PLAN = {
    "primary_endpoint": {
        "name": (
            "patient_level_quadratic_weighted_kappa"
        ),

        "abbreviation": (
            "patient QWK"
        ),

        "direction": (
            "higher is better"
        ),

        "labels": [
            0,
            1,
            2,
            3,
        ],

        "weights": (
            "quadratic"
        ),
    },

    "secondary_patient_endpoints": [
        {
            "name": (
                "patient_level_mean_absolute_error"
            ),

            "abbreviation": (
                "patient MAE"
            ),

            "direction": (
                "lower is better"
            ),
        },

        {
            "name": (
                "patient_level_macro_f1"
            ),

            "abbreviation": (
                "patient macro F1"
            ),

            "direction": (
                "higher is better"
            ),
        },

        {
            "name": (
                "patient_level_accuracy"
            ),

            "abbreviation": (
                "patient accuracy"
            ),

            "direction": (
                "higher is better"
            ),
        },
    ],

    "descriptive_image_endpoints": [
        "image QWK",
        "image MAE",
        "image macro F1",
        "image accuracy",
    ],

    "class_specific_patient_endpoints": [
        "precision for Mayo 0",
        "precision for Mayo 1",
        "precision for Mayo 2",
        "precision for Mayo 3",

        "recall for Mayo 0",
        "recall for Mayo 1",
        "recall for Mayo 2",
        "recall for Mayo 3",

        "F1 for Mayo 0",
        "F1 for Mayo 1",
        "F1 for Mayo 2",
        "F1 for Mayo 3",
    ],

    "required_class_reporting": [
        "exact number of true patients in each Mayo class",
        "patient confusion matrix",
        "image confusion matrix",
    ],
}


# ============================================================
# 19. Predeclare severe-disease sensitivity analysis
# ============================================================

SEVERE_BINARY_SENSITIVITY_PLAN = {
    "role": (
        "predeclared exploratory sensitivity analysis"
    ),

    "positive_class_definition": (
        "Mayo 2 or Mayo 3"
    ),

    "negative_class_definition": (
        "Mayo 0 or Mayo 1"
    ),

    "mapping": {
        "0": 0,
        "1": 0,
        "2": 1,
        "3": 1,
    },

    "metrics": [
        "sensitivity / recall for Mayo 2-3",
        "specificity for Mayo 0-1",
        "binary F1 for Mayo 2-3",
        "balanced accuracy",
        "binary accuracy",
    ],

    "used_for_model_selection": (
        False
    ),

    "used_to_replace_primary_endpoint": (
        False
    ),
}


# ============================================================
# 20. Freeze uncertainty analysis
# ============================================================

BOOTSTRAP_PLAN = {
    "number_of_replicates": (
        5000
    ),

    "random_seed": (
        20260808
    ),

    "sampling_unit": (
        "patient"
    ),

    "sampling_method": (
        "stratified resampling with replacement "
        "within the four primary modal_low MES classes"
    ),

    "confidence_interval": (
        "percentile 95% confidence interval"
    ),

    "confidence_limits": [
        2.5,
        97.5,
    ],

    "primary_model_marginal_intervals": (
        True
    ),

    "comparator_marginal_intervals": (
        True
    ),

    "class_specific_intervals": (
        True
    ),

    "paired_model_comparison": {
        "enabled": (
            True
        ),

        "comparison": (
            "Primary Swin ensemble versus "
            "prespecified ViT ensemble"
        ),

        "same_resampled_patient_indices_for_both_models": (
            True
        ),

        "difference_definition": {
            "QWK": (
                "Swin - ViT"
            ),

            "MAE": (
                "Swin - ViT"
            ),

            "macro_F1": (
                "Swin - ViT"
            ),

            "accuracy": (
                "Swin - ViT"
            ),
        },

        "direction_interpretation": {
            "QWK": (
                "positive favors Swin"
            ),

            "MAE": (
                "negative favors Swin"
            ),

            "macro_F1": (
                "positive favors Swin"
            ),

            "accuracy": (
                "positive favors Swin"
            ),
        },
    },

    "interpretation_note": (
        "Bootstrap intervals quantify uncertainty from "
        "patient resampling conditional on the already "
        "trained checkpoint ensemble. They do not represent "
        "all possible training stochasticity."
    ),
}


# ============================================================
# 21. Freeze test-analysis scope
# ============================================================

ROUTE_A_SCOPE = {
    "route": (
        "Route A"
    ),

    "primary_model": (
        "Swin"
    ),

    "prespecified_comparator": (
        "ViT"
    ),

    "models_in_primary_route_a_test_analysis": [
        "Swin",
        "ViT",
    ],

    "convnext_in_route_a": (
        False
    ),

    "maxvit_in_route_a": (
        False
    ),

    "reason_convnext_maxvit_excluded": (
        "Their revised development-only configuration "
        "selection has not yet been reconstructed."
    ),

    "future_route_b_permitted": (
        True
    ),

    "route_b_must_be_reported_separately": (
        True
    ),

    "route_b_cannot_modify_route_a_primary_results": (
        True
    ),

    "route_b_cannot_reselect_route_a_primary_model": (
        True
    ),
}


# ============================================================
# 22. Freeze prohibited post-test actions
# ============================================================

POST_TEST_PROHIBITIONS = [
    "No model architecture reselection after test access.",
    "No loss-function reselection after test access.",
    "No checkpoint reselection after test access.",
    "No best-seed selection after test access.",
    "No fold-model weighting based on test performance.",
    "No seed weighting based on test performance.",
    "No aggregation-rule change after test access.",
    "No patient-reference-rule change for the primary endpoint.",
    "No threshold optimization on the official test set.",
    "No probability calibration using official test labels.",
    "No test-guided TTA modification.",
    "No test-guided preprocessing modification.",
    "No test-guided ensemble membership modification.",
    "No test-guided exclusion of patients or images.",
    "No replacement of the primary Swin system by an ablation model.",
    "No use of ConvNeXt or MaxViT results to alter Route A conclusions.",
]


# ============================================================
# 23. Freeze exact official-test workflow
# ============================================================

OFFICIAL_TEST_WORKFLOW = [
    {
        "step": 1,

        "action": (
            "Load the official LIMUC test partition only "
            "after this pre-test lock has been written."
        ),
    },

    {
        "step": 2,

        "action": (
            "Audit image paths, image labels, patient IDs, "
            "patient counts, class counts, and within-patient "
            "label heterogeneity without modifying the lock."
        ),
    },

    {
        "step": 3,

        "action": (
            "Derive the primary modal_low patient reference "
            "from official test image-level MES labels."
        ),
    },

    {
        "step": 4,

        "action": (
            "Run all 15 locked Swin checkpoints on every "
            "test image using original + horizontal-flip TTA."
        ),
    },

    {
        "step": 5,

        "action": (
            "Average the 15 Swin probability vectors equally "
            "to obtain one final probability vector per image."
        ),
    },

    {
        "step": 6,

        "action": (
            "Aggregate Swin image probabilities to patients "
            "using mean_probability_argmax."
        ),
    },

    {
        "step": 7,

        "action": (
            "Run the analogous 15-checkpoint ViT comparator "
            "without any model-specific test tuning."
        ),
    },

    {
        "step": 8,

        "action": (
            "Compute all prespecified patient-level and "
            "image-level metrics."
        ),
    },

    {
        "step": 9,

        "action": (
            "Calculate stratified patient-bootstrap 95% "
            "confidence intervals and paired Swin-vs-ViT "
            "bootstrap differences."
        ),
    },

    {
        "step": 10,

        "action": (
            "Compute per-class metrics, confusion matrices, "
            "and the predeclared Mayo 0-1 vs Mayo 2-3 "
            "sensitivity analysis."
        ),
    },

    {
        "step": 11,

        "action": (
            "Perform the predeclared alternative reference "
            "and aggregation analyses as sensitivity analyses "
            "only; these cannot modify the primary result."
        ),
    },

    {
        "step": 12,

        "action": (
            "Save all results without any subsequent "
            "model or analysis reselection."
        ),
    },
]


# ============================================================
# 24. Exact model reconstruction parameters
# ============================================================

MODEL_RECONSTRUCTION_PLAN = {
    "framework": (
        "PyTorch + timm"
    ),

    "num_classes": (
        4
    ),

    "swin": {
        "model_name": (
            SWIN_MODEL_NAME
        ),

        "num_classes": (
            4
        ),

        "pretrained_when_loading_final_checkpoint": (
            False
        ),

        "drop_rate": (
            0.10
        ),

        "drop_path_rate": (
            0.05
        ),

        "load_weights_from": (
            "exact checkpoint paths and SHA256 hashes "
            "in locked_checkpoint_manifest.csv"
        ),
    },

    "vit": {
        "model_name": (
            VIT_MODEL_NAME
        ),

        "num_classes": (
            4
        ),

        "pretrained_when_loading_final_checkpoint": (
            False
        ),

        "drop_rate": (
            0.10
        ),

        "drop_path_rate": (
            0.05
        ),

        "load_weights_from": (
            "exact checkpoint paths and SHA256 hashes "
            "in locked_checkpoint_manifest.csv"
        ),
    },

    "model_mode_during_inference": (
        "eval"
    ),

    "gradient_computation": (
        "disabled"
    ),

    "mixed_precision_inference": (
        True
    ),

    "softmax_dimension": (
        1
    ),
}


# ============================================================
# 25. Construct immutable pre-test lock
# ============================================================

checkpoint_manifest_records = (
    checkpoint_manifest_df[
        [
            "role",
            "backbone_name",
            "model_name",
            "representative_configuration",
            "training_seed",
            "fold",
            "best_epoch",
            "fold_fingerprint",
            "checkpoint_path",
            "checkpoint_sha256",
            "checkpoint_size_bytes",
        ]
    ]
    .to_dict(
        orient="records"
    )
)


source_artifact_records = (
    source_artifact_manifest_df
    .to_dict(
        orient="records"
    )
)


IMMUTABLE_PRETEST_LOCK = {
    "protocol_version": (
        "route_a_final_model_lock_before_test_v1"
    ),

    "status": (
        "PRETEST_LOCKED"
    ),

    "official_test_path_defined": (
        False
    ),

    "official_test_loaded": (
        False
    ),

    "official_test_evaluated": (
        False
    ),

    "training_performed_in_notebook08": (
        False
    ),

    "inference_performed_in_notebook08": (
        False
    ),

    "route_a_scope": (
        ROUTE_A_SCOPE
    ),

    "primary_system": (
        PRIMARY_INFERENCE_SYSTEM
    ),

    "prespecified_comparator_system": (
        COMPARATOR_INFERENCE_SYSTEM
    ),

    "model_reconstruction": (
        MODEL_RECONSTRUCTION_PLAN
    ),

    "evaluation_preprocessing": (
        EVALUATION_PREPROCESSING
    ),

    "patient_reference": (
        PATIENT_REFERENCE_PLAN
    ),

    "patient_aggregation": (
        PATIENT_AGGREGATION_PLAN
    ),

    "metric_plan": (
        METRIC_PLAN
    ),

    "severe_binary_sensitivity": (
        SEVERE_BINARY_SENSITIVITY_PLAN
    ),

    "bootstrap_plan": (
        BOOTSTRAP_PLAN
    ),

    "official_test_workflow": (
        OFFICIAL_TEST_WORKFLOW
    ),

    "post_test_prohibitions": (
        POST_TEST_PROHIBITIONS
    ),

    "checkpoint_manifest": (
        checkpoint_manifest_records
    ),

    "source_artifact_provenance": (
        source_artifact_records
    ),

    "development_dataset_audit": {
        "n_images": (
            int(
                len(
                    development_manifest_df
                )
            )
        ),

        "n_patients": (
            int(
                development_manifest_df[
                    "patient_id"
                ].nunique()
            )
        ),

        "primary_reference_class_counts": {
            str(
                key
            ): int(
                value
            )

            for key, value in (
                development_class_counts.items()
            )
        },

        "fold_patient_counts": {
            str(
                key
            ): int(
                value
            )

            for key, value in (
                development_fold_counts.items()
            )
        },
    },

    "development_evidence_snapshot": {
        "swin_mean_patient_qwk": float(
            swin_summary_row[
                "patient_qwk_mean"
            ]
        ),

        "swin_sd_patient_qwk": float(
            swin_summary_row[
                "patient_qwk_std"
            ]
        ),

        "swin_mean_patient_mae": float(
            swin_summary_row[
                "patient_mae_mean"
            ]
        ),

        "vit_mean_patient_qwk": float(
            vit_summary_row[
                "patient_qwk_mean"
            ]
        ),

        "vit_sd_patient_qwk": float(
            vit_summary_row[
                "patient_qwk_std"
            ]
        ),

        "vit_mean_patient_mae": float(
            vit_summary_row[
                "patient_mae_mean"
            ]
        ),
    },

    "ablation_status": {
        "primary_model_locked_before_ablation": (
            True
        ),

        "ablation_used_for_model_selection": (
            False
        ),

        "primary_model_changed_by_ablation": (
            False
        ),

        "primary_condition": (
            "A0_full_locked"
        ),
    },
}


# ============================================================
# 26. Generate immutable lock fingerprint
# ============================================================

PRETEST_LOCK_FINGERPRINT = (
    dictionary_sha256(
        IMMUTABLE_PRETEST_LOCK
    )
)


FINAL_LOCK = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "pretest_lock_fingerprint_sha256": (
        PRETEST_LOCK_FINGERPRINT
    ),

    **IMMUTABLE_PRETEST_LOCK,
}


FINAL_LOCK_PATH = (
    OUTPUT_ROOT
    /
    "final_model_lock_before_test.json"
)


# ============================================================
# 27. Prevent accidental modification of an existing lock
# ============================================================

if FINAL_LOCK_PATH.exists():
    with open(
        FINAL_LOCK_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        existing_lock = json.load(
            file
        )

    existing_fingerprint = (
        existing_lock.get(
            "pretest_lock_fingerprint_sha256"
        )
    )

    if (
        existing_fingerprint
        !=
        PRETEST_LOCK_FINGERPRINT
    ):
        raise RuntimeError(
            "\nA DIFFERENT PRE-TEST LOCK ALREADY EXISTS.\n\n"
            f"Existing fingerprint:\n"
            f"{existing_fingerprint}\n\n"
            f"Current fingerprint:\n"
            f"{PRETEST_LOCK_FINGERPRINT}\n\n"
            "Do NOT overwrite the old lock. "
            "Investigate the difference before any "
            "official test access."
        )

    print(
        "\nExisting final pre-test lock validated."
    )

    print(
        "Fingerprint:",
        PRETEST_LOCK_FINGERPRINT,
    )

else:
    atomic_json_save(
        FINAL_LOCK,
        FINAL_LOCK_PATH,
    )

    print(
        "\nNEW final pre-test lock written."
    )

    print(
        "Fingerprint:",
        PRETEST_LOCK_FINGERPRINT,
    )


# ============================================================
# 28. Save separate human-readable inference specification
# ============================================================

inference_specification = {
    "pretest_lock_fingerprint_sha256": (
        PRETEST_LOCK_FINGERPRINT
    ),

    "primary_system": (
        PRIMARY_INFERENCE_SYSTEM
    ),

    "comparator_system": (
        COMPARATOR_INFERENCE_SYSTEM
    ),

    "preprocessing": (
        EVALUATION_PREPROCESSING
    ),

    "patient_aggregation": (
        PATIENT_AGGREGATION_PLAN
    ),

    "patient_reference": (
        PATIENT_REFERENCE_PLAN
    ),

    "model_reconstruction": (
        MODEL_RECONSTRUCTION_PLAN
    ),
}


inference_specification_path = (
    OUTPUT_ROOT
    /
    "locked_final_inference_specification.json"
)

atomic_json_save(
    inference_specification,

    inference_specification_path,
)


# ============================================================
# 29. Save statistical-analysis specification
# ============================================================

statistical_analysis_specification = {
    "pretest_lock_fingerprint_sha256": (
        PRETEST_LOCK_FINGERPRINT
    ),

    "metric_plan": (
        METRIC_PLAN
    ),

    "bootstrap_plan": (
        BOOTSTRAP_PLAN
    ),

    "severe_binary_sensitivity_plan": (
        SEVERE_BINARY_SENSITIVITY_PLAN
    ),

    "reference_sensitivity": (
        PATIENT_REFERENCE_PLAN[
            "alternative_reference_sensitivity"
        ]
    ),

    "aggregation_sensitivity": (
        PATIENT_AGGREGATION_PLAN[
            "alternative_aggregation_sensitivity"
        ]
    ),
}


statistical_analysis_path = (
    OUTPUT_ROOT
    /
    "locked_official_test_statistical_analysis_plan.json"
)

atomic_json_save(
    statistical_analysis_specification,

    statistical_analysis_path,
)


# ============================================================
# 30. Save compact model table
# ============================================================

model_table_df = pd.DataFrame([
    {
        "role": (
            "Primary"
        ),

        "backbone": (
            "Swin"
        ),

        "model_name": (
            SWIN_MODEL_NAME
        ),

        "configuration": (
            EXPECTED_PRIMARY_CONFIGURATION
        ),

        "ordinal_alpha": (
            SWIN_CONFIG[
                "ordinal_alpha"
            ]
        ),

        "ordinal_lambda": (
            SWIN_CONFIG[
                "ordinal_loss_weight"
            ]
        ),

        "n_seeds": (
            3
        ),

        "n_folds": (
            5
        ),

        "n_checkpoints": (
            15
        ),

        "checkpoint_weighting": (
            "equal"
        ),

        "patient_reference": (
            PRIMARY_REFERENCE_RULE
        ),

        "patient_aggregation": (
            LOCKED_PATIENT_AGGREGATION
        ),
    },

    {
        "role": (
            "Prespecified comparator"
        ),

        "backbone": (
            "ViT"
        ),

        "model_name": (
            VIT_MODEL_NAME
        ),

        "configuration": (
            EXPECTED_COMPARATOR_CONFIGURATION
        ),

        "ordinal_alpha": (
            VIT_CONFIG[
                "ordinal_alpha"
            ]
        ),

        "ordinal_lambda": (
            VIT_CONFIG[
                "ordinal_loss_weight"
            ]
        ),

        "n_seeds": (
            3
        ),

        "n_folds": (
            5
        ),

        "n_checkpoints": (
            15
        ),

        "checkpoint_weighting": (
            "equal"
        ),

        "patient_reference": (
            PRIMARY_REFERENCE_RULE
        ),

        "patient_aggregation": (
            LOCKED_PATIENT_AGGREGATION
        ),
    },
])


model_table_path = (
    OUTPUT_ROOT
    /
    "locked_route_a_model_table.csv"
)

atomic_csv_save(
    model_table_df,

    model_table_path,

    index=False,
)


# ============================================================
# 31. Save lock receipt
# ============================================================

LOCK_RECEIPT = f"""
ROUTE A PRE-TEST LOCK RECEIPT
=============================

Created UTC:
{FINAL_LOCK["created_utc"]}

PRE-TEST LOCK SHA256 FINGERPRINT:
{PRETEST_LOCK_FINGERPRINT}

STATUS:
PRETEST_LOCKED

PRIMARY MODEL:
Swin

PRIMARY MODEL IDENTIFIER:
{SWIN_MODEL_NAME}

PRIMARY CONFIGURATION:
{EXPECTED_PRIMARY_CONFIGURATION}

PRIMARY FINAL INFERENCE SYSTEM:
Equal-probability ensemble of 15 checkpoints
(3 seeds x 5 folds).

COMPARATOR:
ViT

COMPARATOR MODEL IDENTIFIER:
{VIT_MODEL_NAME}

COMPARATOR CONFIGURATION:
{EXPECTED_COMPARATOR_CONFIGURATION}

COMPARATOR INFERENCE SYSTEM:
Equal-probability ensemble of 15 checkpoints
(3 seeds x 5 folds).

TEST-TIME AUGMENTATION:
Original image + horizontal flip.
The two probability vectors are averaged within checkpoint.

IMAGE PREPROCESSING:
Resize directly to 384 x 384.
RGB.
ImageNet normalization.
No random augmentation.

PRIMARY PATIENT REFERENCE:
modal_low

PRIMARY PATIENT AGGREGATION:
mean_probability_argmax

PRIMARY ENDPOINT:
Patient-level quadratic weighted kappa.

SECONDARY PATIENT ENDPOINTS:
MAE
Macro F1
Accuracy

BOOTSTRAP:
5000 stratified patient bootstrap replicates.
Seed = 20260808.

PAIRED MODEL COMPARISON:
Swin - ViT using identical bootstrap patient samples.

ROUTE A MODELS:
Swin
ViT

CONVNEXT IN ROUTE A:
No

MAXVIT IN ROUTE A:
No

OFFICIAL TEST PATH DEFINED IN NOTEBOOK 08:
No

OFFICIAL TEST DATA LOADED IN NOTEBOOK 08:
No

OFFICIAL TEST DATA EVALUATED IN NOTEBOOK 08:
No

TRAINING PERFORMED IN NOTEBOOK 08:
No

IMPORTANT:
After official test access, no model, checkpoint, seed,
fold weight, aggregation rule, patient-reference rule,
preprocessing rule, TTA rule, threshold, calibration,
or primary endpoint may be changed on the basis of
official test performance.
"""


lock_receipt_path = (
    OUTPUT_ROOT
    /
    "PRETEST_LOCK_RECEIPT.txt"
)

atomic_text_save(
    LOCK_RECEIPT.strip()
    +
    "\n",

    lock_receipt_path,
)


# ============================================================
# 32. Save READY flag
# ============================================================

ready_flag_text = f"""
READY FOR LOCKED OFFICIAL TEST EVALUATION

Pre-test lock fingerprint:
{PRETEST_LOCK_FINGERPRINT}

This flag means:
- development-only model selection is complete;
- Route A primary model is frozen;
- Route A comparator is frozen;
- exact checkpoint files are hashed;
- inference procedure is frozen;
- primary reference is frozen;
- aggregation is frozen;
- statistical analysis is frozen;
- official test data have NOT yet been loaded.

The next notebook may open the official test partition,
but it must obey final_model_lock_before_test.json exactly.
"""


ready_flag_path = (
    OUTPUT_ROOT
    /
    "READY_FOR_LOCKED_OFFICIAL_TEST.txt"
)

atomic_text_save(
    ready_flag_text.strip()
    +
    "\n",

    ready_flag_path,
)


# ============================================================
# 33. Save lock-integrity manifest
# ============================================================

LOCK_OUTPUT_FILES_TO_HASH = [
    FINAL_LOCK_PATH,
    checkpoint_manifest_path,
    source_artifact_manifest_path,
    inference_specification_path,
    statistical_analysis_path,
    model_table_path,
    lock_receipt_path,
    ready_flag_path,
]


lock_output_integrity_rows = []

for path in LOCK_OUTPUT_FILES_TO_HASH:
    lock_output_integrity_rows.append({
        "filename": (
            path.name
        ),

        "path": str(
            path
        ),

        "sha256": (
            file_sha256(
                path
            )
        ),

        "size_bytes": (
            int(
                path.stat().st_size
            )
        ),
    })


lock_integrity_df = pd.DataFrame(
    lock_output_integrity_rows
)


lock_integrity_manifest_path = (
    OUTPUT_ROOT
    /
    "pretest_lock_output_integrity_manifest.csv"
)

atomic_csv_save(
    lock_integrity_df,

    lock_integrity_manifest_path,

    index=False,
)


# ============================================================
# 34. Environment metadata
# ============================================================

environment_metadata = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": (
        platform.python_version()
    ),

    "platform": (
        platform.platform()
    ),

    "numpy": (
        np.__version__
    ),

    "pandas": (
        pd.__version__
    ),

    "pretest_lock_fingerprint_sha256": (
        PRETEST_LOCK_FINGERPRINT
    ),

    "official_test_path_defined": (
        False
    ),

    "official_test_loaded": (
        False
    ),

    "official_test_evaluated": (
        False
    ),
}


environment_metadata_path = (
    OUTPUT_ROOT
    /
    "environment_metadata.json"
)

atomic_json_save(
    environment_metadata,

    environment_metadata_path,
)


# ============================================================
# 35. Display exact checkpoint summary
# ============================================================

checkpoint_summary_df = (
    checkpoint_manifest_df
    .groupby(
        [
            "role",
            "backbone_name",
            "model_name",
            "representative_configuration",
        ],

        as_index=False,
    )
    .agg(
        n_checkpoints=(
            "checkpoint_path",
            "count",
        ),

        n_seeds=(
            "training_seed",
            "nunique",
        ),

        n_folds=(
            "fold",
            "nunique",
        ),

        total_checkpoint_bytes=(
            "checkpoint_size_bytes",
            "sum",
        ),
    )
)


print("\n" + "=" * 125)
print("LOCKED ROUTE A SYSTEMS")
print("=" * 125)

display(
    model_table_df
)


print("\n" + "=" * 125)
print("LOCKED CHECKPOINT MANIFEST SUMMARY")
print("=" * 125)

display(
    checkpoint_summary_df
)


# ============================================================
# 36. Display exact pre-test endpoint plan
# ============================================================

metric_display_df = pd.DataFrame([
    {
        "analysis_role": (
            "Primary"
        ),

        "endpoint": (
            "Patient QWK"
        ),

        "direction": (
            "Higher is better"
        ),
    },

    {
        "analysis_role": (
            "Secondary"
        ),

        "endpoint": (
            "Patient MAE"
        ),

        "direction": (
            "Lower is better"
        ),
    },

    {
        "analysis_role": (
            "Secondary"
        ),

        "endpoint": (
            "Patient macro F1"
        ),

        "direction": (
            "Higher is better"
        ),
    },

    {
        "analysis_role": (
            "Secondary"
        ),

        "endpoint": (
            "Patient accuracy"
        ),

        "direction": (
            "Higher is better"
        ),
    },

    {
        "analysis_role": (
            "Descriptive"
        ),

        "endpoint": (
            "Image QWK / MAE / macro F1 / accuracy"
        ),

        "direction": (
            "Reported descriptively"
        ),
    },

    {
        "analysis_role": (
            "Exploratory sensitivity"
        ),

        "endpoint": (
            "Mayo 0-1 vs Mayo 2-3"
        ),

        "direction": (
            "Predeclared, not used for model selection"
        ),
    },
])


print("\n" + "=" * 125)
print("PREDECLARED OFFICIAL-TEST ENDPOINTS")
print("=" * 125)

display(
    metric_display_df
)


# ============================================================
# 37. Display immutable prohibitions
# ============================================================

print("\n" + "=" * 125)
print("POST-TEST CHANGES THAT ARE NOW PROHIBITED")
print("=" * 125)

for index, prohibition in enumerate(
    POST_TEST_PROHIBITIONS,
    start=1,
):
    print(
        f"{index:02d}. {prohibition}"
    )


# ============================================================
# 38. Final pre-test lock verification
# ============================================================

# Re-read the final lock from disk and verify fingerprint.
with open(
    FINAL_LOCK_PATH,
    "r",
    encoding="utf-8",
) as file:
    disk_lock = json.load(
        file
    )


disk_fingerprint = (
    disk_lock[
        "pretest_lock_fingerprint_sha256"
    ]
)


if (
    disk_fingerprint
    !=
    PRETEST_LOCK_FINGERPRINT
):
    raise RuntimeError(
        "Final lock fingerprint failed disk verification."
    )


# Verify exact checkpoint count one last time.
if (
    len(
        disk_lock[
            "checkpoint_manifest"
        ]
    )
    !=
    30
):
    raise RuntimeError(
        "Final lock does not contain exactly 30 checkpoints."
    )


primary_locked_checkpoint_count = sum(
    1
    for record in disk_lock[
        "checkpoint_manifest"
    ]
    if record[
        "role"
    ]
    ==
    "primary"
)


comparator_locked_checkpoint_count = sum(
    1
    for record in disk_lock[
        "checkpoint_manifest"
    ]
    if record[
        "role"
    ]
    ==
    "prespecified_comparator"
)


if (
    primary_locked_checkpoint_count
    !=
    15
):
    raise RuntimeError(
        "Primary Swin lock does not contain 15 checkpoints."
    )


if (
    comparator_locked_checkpoint_count
    !=
    15
):
    raise RuntimeError(
        "ViT comparator lock does not contain 15 checkpoints."
    )


# ============================================================
# 39. FINAL STATUS
# ============================================================

print("\n" + "=" * 125)
print("08_final_model_lock_before_test COMPLETED SUCCESSFULLY")
print("=" * 125)

print(
    "\nPRE-TEST LOCK STATUS:",
    "LOCKED",
)

print(
    "\nPRE-TEST LOCK FINGERPRINT:"
)

print(
    PRETEST_LOCK_FINGERPRINT
)


print(
    "\nPrimary system:"
)

print(
    "  Swin"
)

print(
    "  ",
    SWIN_MODEL_NAME,
)

print(
    "  15 checkpoints = 3 seeds x 5 folds"
)

print(
    "  Equal probability averaging"
)

print(
    "  Original + horizontal-flip TTA"
)


print(
    "\nPrespecified comparator:"
)

print(
    "  ViT"
)

print(
    "  ",
    VIT_MODEL_NAME,
)

print(
    "  15 checkpoints = 3 seeds x 5 folds"
)

print(
    "  Equal probability averaging"
)

print(
    "  Original + horizontal-flip TTA"
)


print(
    "\nPrimary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Primary patient aggregation:",
    LOCKED_PATIENT_AGGREGATION,
)

print(
    "Primary endpoint:",
    "patient-level QWK",
)

print(
    "Bootstrap:",
    "5,000 stratified patient replicates",
)


print(
    "\nRoute A includes:"
)

print(
    "  Swin primary"
)

print(
    "  ViT comparator"
)

print(
    "  ConvNeXt: excluded"
)

print(
    "  MaxViT: excluded"
)


print(
    "\nOfficial test path defined:",
    False,
)

print(
    "Official test data loaded:",
    False,
)

print(
    "Official test data evaluated:",
    False,
)

print(
    "Training performed:",
    False,
)

print(
    "Inference performed:",
    False,
)


print(
    "\nMain lock file:"
)

print(
    FINAL_LOCK_PATH
)


print(
    "\nExact checkpoint manifest:"
)

print(
    checkpoint_manifest_path
)


print(
    "\nInference specification:"
)

print(
    inference_specification_path
)


print(
    "\nStatistical-analysis specification:"
)

print(
    statistical_analysis_path
)


print(
    "\nLock receipt:"
)

print(
    lock_receipt_path
)


print(
    "\nReady flag:"
)

print(
    ready_flag_path
)


print(
    "\nNEXT STEP:"
)

print(
    "Only after preserving this lock may Notebook 09 "
    "define and open the official LIMUC test partition."
)

print(
    "Notebook 09 must verify this fingerprint and all "
    "checkpoint hashes before performing test inference."
)

print(
    "\nDO NOT MODIFY THE LOCK AFTER OFFICIAL TEST ACCESS."
)